# Data Cleaning -- Pembrolizumab + Chemotherapy vs. Pembrolizumab in Positive PD-L1
**This notebook prepares Flatiron Health CSV files for patients with advanced head and neck cancer treated with first-line pembrolizumab plus platinum-based chemotherapy or pembrolizumab. Patients with positive or unknown PD-L1 results are included. Each CSV is cleaned using the flatiron_cleaner package. The cleaned dataframes are then merged into a single dataset, which will be used for survival analysis.**

## Import packages

In [1]:
import numpy as np
import pandas as pd

from flatiron_cleaner import DataProcessorHeadNeck, merge_dataframes

## Import data

In [2]:
df = pd.read_csv('../outputs/pembrochemo_pembro_index.csv')

In [3]:
df.head(3)

,PatientID,LineName,StartDate
0,F33D856BE4FC6,pembro_platinum,2023-01-05
1,F2C55497C35F5,pembro_platinum,2021-10-18
2,F51465980052A,pembro_platinum,2019-05-10


In [4]:
df.shape

(1854, 3)

## Clean CSV files 

In [5]:
# Initialize class 
processor = DataProcessorHeadNeck()

### Process Enhanced_AdvHeadNeck.csv

In [6]:
enhanced_df = processor.process_enhanced(file_path = '../data/Enhanced_AdvHeadNeck.csv',
                                         index_date_df = df,
                                         index_date_column = 'StartDate',
                                         drop_dates = False)

2026-01-19 21:25:35,323 - INFO - Successfully read Enhanced_AdvHeadNeck.csv file with shape: (11347, 15) and unique PatientIDs: 11347
2026-01-19 21:25:35,328 - INFO - Successfully filtered Enhanced_AdvHeadNeck.csv file with shape: (1854, 16) and unique PatientIDs: 1854
2026-01-19 21:25:35,340 - INFO - Successfully processed Enhanced_AdvHeadNeck.csv file with final shape: (1854, 20) and unique PatientIDs: 1854


In [7]:
enhanced_df['days_adv_to_treatment'] = (enhanced_df['imported_StartDate'] - enhanced_df['AdvancedDiagnosisDate']).dt.days
enhanced_df['days_adv_to_treatment'] = np.where(enhanced_df['days_adv_to_treatment'] < 0 , 0, enhanced_df['days_adv_to_treatment'])
enhanced_df['days_to_treatment_before_30d'] = np.where(enhanced_df['days_adv_to_treatment'] < 30, 1, 0)

In [8]:
enhanced_df.GroupStage_mod.value_counts(dropna = False)

GroupStage_mod
IV_locoregional    695
III                291
unknown            229
II                 220
IV_metastatic      193
I                  142
IV_NOS              84
Name: count, dtype: int64

In [9]:
enhanced_df['GroupStage_mod'] = enhanced_df["GroupStage_mod"].map({
    '0': 0, 
    'I': 1,
    'II': 2,
    'III': 3,
    'IV_NOS': 4,
    'IV_locoregional': 4,
    'IV_metastatic': 5, 
    'unknown': np.nan
})

enhanced_df['GroupStage_mod_na'] = np.where(enhanced_df['GroupStage_mod'].isna(), 1, 0)

# impute 4 since stage IV (locoregional) is most common
enhanced_df['GroupStage_mod'] = enhanced_df['GroupStage_mod'].fillna(4)

In [10]:
enhanced_df['adv_diagnosis_year'] = pd.to_numeric(enhanced_df['adv_diagnosis_year'])
enhanced_df['before_2020'] = np.where(enhanced_df['adv_diagnosis_year'] < 2020, 1, 0)

In [11]:
enhanced_df.SmokingStatus.value_counts(dropna = False)

SmokingStatus
History of smoking       1406
No history of smoking     448
Name: count, dtype: int64

In [12]:
enhanced_df['SmokingStatus'] = (
    enhanced_df['SmokingStatus']
    .map({'History of smoking': 1, 'No history of smoking': 0})
    .astype('Int64')
)

In [13]:
enhanced_df.HPVStatus_mod.value_counts(dropna = False)

HPVStatus_mod
positive    716
negative    686
NaN         421
unknown      31
Name: count, dtype: int64

In [14]:
enhanced_df['HPVStatus_mod'] = enhanced_df['HPVStatus_mod'].fillna('unknown') 

In [15]:
enhanced_df = enhanced_df.drop(columns = ['DiagnosisDate', 
                                          'AdvancedDiagnosisDate', 
                                          'imported_StartDate',
                                          'FirstLocalRecurDate',
                                          'FirstDistantRecurDate',
                                          'PrimarySurgeryDate',
                                          'PrimaryRadiationDate'])

### Process Demographics.csv 

In [16]:
demographics_df = processor.process_demographics(file_path = '../data/Demographics.csv',
                                                 index_date_df = df,
                                                 index_date_column = 'StartDate')

2026-01-19 21:25:35,381 - INFO - Successfully read Demographics.csv file with shape: (11347, 6) and unique PatientIDs: 11347
2026-01-19 21:25:35,388 - INFO - Successfully processed Demographics.csv file with final shape: (1854, 6) and unique PatientIDs: 1854


In [17]:
demographics_df.Gender.value_counts(dropna = False)

Gender
M    1457
F     397
Name: count, dtype: int64

In [18]:
# Impute missing with most common sex (male)
demographics_df['sex_male'] = np.where(demographics_df['Gender'] == 'F', 0, 1)

In [19]:
demographics_df = demographics_df.drop(columns = ['Gender'])

### Process Enhanced_AdvHeadNeckBiomarkers.csv

In [20]:
biomarkers_df = processor.process_biomarkers(file_path = '../data/Enhanced_AdvHeadNeckBiomarkers.csv',
                                             index_date_df = df, 
                                             index_date_column = 'StartDate',
                                             days_before = None, 
                                             days_after = 30,
                                             pdl1_result_type = 'cps')

2026-01-19 21:25:35,409 - INFO - Successfully read Enhanced_AdvHeadNeckBiomarkers.csv file with shape: (4379, 17) and unique PatientIDs: 3131
2026-01-19 21:25:35,414 - INFO - Successfully merged Enhanced_AdvHeadNeckBiomarkers.csv df with index_date_df resulting in shape: (1761, 18) and unique PatientIDs: 1229
2026-01-19 21:25:35,417 - WARNING - Found 2 records (PatientIDs: ['FA92F4315E170', 'F62450C3FE8C1']) with PD-L1 positive but CPS 0 or <1 - possible data quality issue
2026-01-19 21:25:35,459 - INFO - Successfully processed Enhanced_AdvHeadNeckBiomarkers.csv file with final shape: (1854, 3) and unique PatientIDs: 1854


In [21]:
biomarkers_df.PDL1_status.value_counts(dropna = False)

PDL1_status
NaN         796
unknown     636
positive    311
negative    111
Name: count, dtype: int64

In [22]:
biomarkers_df['PDL1_status'] = biomarkers_df['PDL1_status'].fillna('unknown')

### Process ECOG.csv

In [23]:
ecog_df = processor.process_ecog(file_path = '../data/ECOG.csv', 
                                 index_date_df = df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 days_before_further = 180)

2026-01-19 21:25:35,539 - INFO - Successfully read ECOG.csv file with shape: (199312, 4) and unique PatientIDs: 8957
2026-01-19 21:25:35,569 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (42871, 5) and unique PatientIDs: 1651
2026-01-19 21:25:35,600 - INFO - Successfully processed ECOG.csv file with final shape: (1854, 3) and unique PatientIDs: 1854


In [24]:
ecog_df.ecog_index.value_counts(dropna = False)

ecog_index
1      712
NaN    443
0      442
2      210
3       42
4        5
Name: count, dtype: int64

In [25]:
ecog_df['ecog_index'] = ecog_df['ecog_index'].astype('float64')
ecog_df['ecog_index_na'] = np.where(ecog_df['ecog_index'].isna(), 1, 0)

# impute 1 for missing ECOG since most common
ecog_df['ecog_index'] = ecog_df['ecog_index'].fillna(1)

In [26]:
ecog_df['ecog_newly_gte2'] = ecog_df['ecog_newly_gte2'].fillna(0)

### Process Vitals.csv

In [27]:
vitals_df = processor.process_vitals(file_path = '../data/Vitals.csv',
                                     index_date_df = df,
                                     index_date_column = 'StartDate',
                                     weight_days_before = 90,
                                     days_after = 0,
                                     vital_summary_lookback = 180, 
                                     abnormal_reading_threshold = 1)

2026-01-19 21:25:39,192 - INFO - Successfully read Vitals.csv file with shape: (4024923, 16) and unique PatientIDs: 11339
2026-01-19 21:25:40,597 - INFO - Successfully merged Vitals.csv df with index_date_df resulting in shape: (780706, 17) and unique PatientIDs: 1854
2026-01-19 21:25:40,919 - INFO - Successfully processed Vitals.csv file with final shape: (1854, 8) and unique PatientIDs: 1854


### Process Lab.csv

In [28]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 summary_lookback = 180)

2026-01-19 21:25:51,877 - INFO - Successfully read Lab.csv file with shape: (9064402, 17) and unique PatientIDs: 10989
2026-01-19 21:25:53,674 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (1730206, 18) and unique PatientIDs: 1840
2026-01-19 21:25:57,717 - INFO - Successfully processed Lab.csv file with final shape: (1854, 76) and unique PatientIDs: 1854


### Process MedicationAdministration.csv

In [29]:
medications_df = processor.process_medications(file_path = '../data/MedicationAdministration.csv',
                                               index_date_df = df,
                                               index_date_column = 'StartDate',
                                               days_before = 90,
                                               days_after = 0)

2026-01-19 21:25:59,185 - INFO - Successfully read MedicationAdministration.csv file with shape: (1098540, 11) and unique PatientIDs: 9978
2026-01-19 21:25:59,499 - INFO - Successfully merged MedicationAdministration.csv df with index_date_df resulting in shape: (200453, 12) and unique PatientIDs: 1838
2026-01-19 21:25:59,532 - INFO - Successfully processed MedicationAdministration.csv file with final shape: (1854, 9) and unique PatientIDs: 1854


### Process Diagnosis.csv

In [30]:
diagnosis_df = processor.process_diagnosis(file_path = '../data/Diagnosis.csv',
                                           index_date_df = df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0)

2026-01-19 21:26:00,042 - INFO - Successfully read Diagnosis.csv file with shape: (696197, 6) and unique PatientIDs: 11347
2026-01-19 21:26:00,134 - INFO - Successfully merged Diagnosis.csv df with index_date_df resulting in shape: (137246, 7) and unique PatientIDs: 1854
2026-01-19 21:26:00,577 - INFO - Successfully processed Diagnosis.csv file with final shape: (1854, 38) and unique PatientIDs: 1854


### Process Enhanced_Mortality_V2.csv

In [31]:
mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv', 
                                           biomarkers_path = '../data/Enhanced_AdvHeadNeckBiomarkers.csv', 
                                           drop_dates = False)

2026-01-19 21:26:00,588 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (7857, 2) and unique PatientIDs: 7857
2026-01-19 21:26:00,597 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (1854, 3) and unique PatientIDs: 1854
2026-01-19 21:26:00,961 - INFO - The following columns ['last_visit_date', 'last_biomarker_date'] are used to calculate the last EHR date
2026-01-19 21:26:00,965 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (1854, 6) and unique PatientIDs: 1854. There are 0 out of 1854 patients with missing duration values


In [32]:
mortality_df = mortality_df[['PatientID', 'event', 'duration']]

In [33]:
mortality_df = mortality_df.query('duration >= 0')

### Process Insurance.csv

In [34]:
insurance_df = processor.process_insurance(file_path = '../data/Insurance.csv',
                                           index_date_df = df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0,
                                           missing_date_strategy = 'liberal')

2026-01-19 21:26:01,059 - INFO - Successfully read Insurance.csv file with shape: (53554, 14) and unique PatientIDs: 10922
2026-01-19 21:26:01,092 - INFO - Successfully merged Insurance.csv df with index_date_df resulting in shape: (10048, 15) and unique PatientIDs: 1841
2026-01-19 21:26:01,123 - INFO - Successfully processed Insurance.csv file with final shape: (1854, 5) and unique PatientIDs: 1854


### Process SocialDeterminantsOfHealth.csv 

In [35]:
ses_df = pd.read_csv('../data/SocialDeterminantsOfHealth.csv')

In [36]:
ses_df.head(3)

,PatientID,SESIndex2015_2019
0,F7449ACE55D66,4
1,F169CCBD5AA1C,3
2,F3C7FCD3F73B4,4


In [37]:
ses_df.SESIndex2015_2019.value_counts(dropna = False)

SESIndex2015_2019
3                  2298
2                  2184
4                  2159
1 - Lowest SES     2044
5 - Highest SES    1581
NaN                1081
Name: count, dtype: int64

In [38]:
ses_df['ses_mod'] = ses_df['SESIndex2015_2019'].map({
    '1 - Lowest SES' : 1, 
    '2': 2, 
    '3': 3, 
    '4': 4, 
    '5 - Highest SES': 5
})

ses_df['ses_mod_na'] = np.where(ses_df['ses_mod'].isna(), 1, 0)

# impute 3 for missing SES
ses_df['ses_mod'] = ses_df['ses_mod'].fillna(3)

In [39]:
ses_df = ses_df[['PatientID', 'ses_mod', 'ses_mod_na']]

In [40]:
ses_df = ses_df[ses_df.PatientID.isin(df.PatientID)]

## Merge dataframes

In [41]:
pembrochemo_pembro_features_df = merge_dataframes(enhanced_df,
                                                  demographics_df,
                                                  biomarkers_df,
                                                  ecog_df,
                                                  vitals_df,
                                                  labs_df,
                                                  medications_df,
                                                  diagnosis_df, 
                                                  mortality_df, 
                                                  insurance_df,
                                                  ses_df,
                                                  merge_type = 'inner')

2026-01-19 21:26:01,151 - INFO - Anticipated number of merges: 10
2026-01-19 21:26:01,151 - INFO - Anticipated number of columns in final dataframe presuming all columns are unique except for PatientID: 162
2026-01-19 21:26:01,152 - INFO - Dataset 1 shape: (1854, 17), unique PatientIDs: 1854
2026-01-19 21:26:01,153 - INFO - Dataset 2 shape: (1854, 6), unique PatientIDs: 1854
2026-01-19 21:26:01,153 - INFO - Dataset 3 shape: (1854, 3), unique PatientIDs: 1854
2026-01-19 21:26:01,154 - INFO - Dataset 4 shape: (1854, 4), unique PatientIDs: 1854
2026-01-19 21:26:01,154 - INFO - Dataset 5 shape: (1854, 8), unique PatientIDs: 1854
2026-01-19 21:26:01,155 - INFO - Dataset 6 shape: (1854, 76), unique PatientIDs: 1854
2026-01-19 21:26:01,155 - INFO - Dataset 7 shape: (1854, 9), unique PatientIDs: 1854
2026-01-19 21:26:01,156 - INFO - Dataset 8 shape: (1854, 38), unique PatientIDs: 1854
2026-01-19 21:26:01,156 - INFO - Dataset 9 shape: (1847, 3), unique PatientIDs: 1847
2026-01-19 21:26:01,156 -

In [42]:
pembrochemo_pembro_features_df.shape

(1847, 162)

In [43]:
pembrochemo_pembro_features_df.PDL1_status.value_counts(normalize = True)

PDL1_status
unknown     0.771521
positive    0.168381
negative    0.060097
Name: proportion, dtype: float64

In [44]:
pembrochemo_pembro_features_df = pembrochemo_pembro_features_df.query('PDL1_status != "negative"')

In [45]:
pembrochemo_pembro_features_df.shape

(1736, 162)

In [46]:
pembrochemo_pembro_features_df.head(2)

,PatientID,AdvancedDiagnosisCriteria,PrimarySite,SmokingStatus,HPVTested,GroupStage_mod,HPVStatus_mod,received_surgery,received_radiation,had_local_recurrence,...,other_gi_met,other_met,event,duration,commercial,medicaid,medicare,other_insurance,ses_mod,ses_mod_na
0,F33D856BE4FC6,Distant recurrence,Oropharynx,1,Yes,1.0,positive,0,1,1,...,0,1,1,100.0,1,1,1,1,3.0,0
1,FEBCC4F65CC4C,Second locoregional recurrence,Oropharynx,0,Yes,3.0,positive,0,1,1,...,0,0,0,1137.0,1,0,1,0,3.0,0


## Export dataframe

In [47]:
pembrochemo_pembro_features_df.to_csv('../outputs/pembrochemo_pembro_features_df.csv', index = False)

In [48]:
# Save dtypes
pembrochemo_pembro_features_df.dtypes.apply(lambda x: x.name).to_csv('../outputs/pembrochemo_pembro_features_dtypes.csv')